# Local AI Chatbot Notebook
In this step, we install the required packages and ensure Ollama is accessible.
Make sure you have Ollama installed on your machine and running locally (`ollama serve`).

In [11]:
# !pip install ollama

## Step 1: Connect to Local Ollama Instance
We verify that Python can connect to the local Ollama server (running by default at `http://localhost:11434`) and list available models.

In [12]:
import ollama

# List all the available pulled models locally
models = ollama.list()
print("Available models -> ")
for model in models.models:
    print(f"- {model.model}")

    

Available models -> 
- nomic-embed-text:latest
- qwen2.5:3b


In [13]:
# Set your target model name here (e.g., 'llama3.2', 'mistral', 'gemma2')
MODEL_NAME = "qwen2.5:3b"  # Change this to match a model installed on your system

## Step 2: Single-Turn Text Completion
A simple request where we pass a prompt and wait for the entire response before displaying it.

In [14]:
response = ollama.chat(
    model = MODEL_NAME,
    messages=[
        {"role": "user", "content": "Explain quantum computing in one simple sentence."}
    ]
    
)

In [15]:
# Extract response text
answer = response.message.content
print("Bot Response:\n", answer)

Bot Response:
 Quantum computing uses the properties of quantum mechanics to process information using quantum bits (qubits) that can be in multiple states simultaneously, potentially solving complex problems much faster than classical computers.


## Step 3: Streaming Tokens Real-Time
To avoid long pauses for long generated answers, set `stream=True` to print tokens as they generate.

In [16]:
stream = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a 3-paragraph essay about deep-sea exploration."}
    ],
    stream=True  # Enables real-time streaming
)

print("Bot Response (Streaming):\n")
for chunk in stream:
    # Print tokens immediately without adding newlines
    print(chunk.message.content, end="", flush=True)

Bot Response (Streaming):

Deep-sea exploration is an area of study that has captivated human curiosity for centuries, driven by the allure of mysteries hidden beneath the ocean's surface and the potential scientific discoveries waiting to be unearthed. The depths of the ocean extend over a thousand meters below sea level and continue downwards into the dark abyss known as the hadal zone, where water pressure can reach thousands of times greater than at the surface. Despite this formidable environment, deep-sea exploration has revealed ecosystems incredibly adapted to these extreme conditions, including hydrothermal vents teeming with unique life forms such as tubeworms and giant clams that thrive on the chemical energy produced by the ocean's geological processes.

One of the most significant milestones in deep-sea exploration occurred when humans first reached the Mariana Trench, the deepest part of the world’s oceans. In 1960, Swiss engineer Jacques Picqué became the first person to

## Step 4: Multi-Turn Chat Memory
LLMs are stateless by default. To maintain conversation context, we append user and assistant messages into a structured array.

In [17]:
# Initialize message history with an optional system prompt
conversation_history = [
    {"role": "system", "content": "You are a concise and helpful python coding assistant."}
]

def send_message(user_input, history, model=MODEL_NAME):
    # 1. Append User Input
    history.append({"role": "user", "content": user_input})
    
    # 2. Get Response from Ollama
    stream = ollama.chat(model=model, messages=history, stream=True)
    
    full_response = ""
    print(f"User: {user_input}\nAssistant: ", end="", flush=True)
    
    # 3. Stream and accumulate text
    for chunk in stream:
        token = chunk.message.content
        print(token, end="", flush=True)
        full_response += token # type: ignore
    print("\n" + "-"*50)
    
    # 4. Append Assistant Response back to history so memory persists
    history.append({"role": "assistant", "content": full_response})

# --- Test Multi-Turn Memory ---
send_message("Hi, my name is Alex and I want to learn Python.", conversation_history)
send_message("What is my name and what do I want to learn?", conversation_history)

User: Hi, my name is Alex and I want to learn Python.
Assistant: Hello Alex! That's great that you're looking to learn Python. It's a powerful language with many applications in web development, data science, artificial intelligence, and more.

If you're just starting out, here are some steps you can take:

1. **Learn the Basics**: Start by learning about variables, data types (like strings, numbers, lists), control structures (if-else statements, loops like for and while), functions, and basic syntax.

2. **Practice Coding**: Try to solve simple problems or create small programs to get used to Python coding.

3. **Use Online Resources**:
   - Websites: Codecademy, freeCodeCamp.org
   - Books: "Python Crash Course" by Eric Matthes is a good beginner's book.
   - YouTube Channels: Corey Schafer (YouTube), Traversy Media are great for learning Python.

4. **Join Communities**: Participate in forums like Stack Overflow or join communities on Reddit where you can ask questions and share yo

## Step 5: Interactive Chatbot Terminal Interface
Run this cell to launch an interactive session inside the notebook. Type `quit` or `exit` to end the session.

In [19]:
def start_interactive_chat(model=MODEL_NAME):
    history = [
        {"role": "system", "content": "You are a friendly and polite AI assistant."}
    ]
    
    print(f"--- Chatbot session started ({model}). Type 'exit' to quit. ---")
    
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ["exit", "quit", "bye"]:
            print("Chat session ended.")
            break
            
        if not user_input:
            continue
            
        # Append message and execute request
        history.append({"role": "user", "content": user_input})
        
        print("Bot: ", end="", flush=True)
        stream = ollama.chat(model=model, messages=history, stream=True)
        
        bot_response = ""
        for chunk in stream:
            token = chunk.message.content
            print(token, end="", flush=True)
            bot_response += token # type: ignore
        print("\n")
        
        # Save assistant context
        history.append({"role": "assistant", "content": bot_response})

# Uncomment below to launch the live CLI loop inside notebook:
start_interactive_chat()

--- Chatbot session started (qwen2.5:3b). Type 'exit' to quit. ---
Bot: Hello! That's okay, it's good to know you've encountered an issue. How can I assist you in resolving this problem? Do you mind sharing more details about what the problem is so I can offer better help or advice?

Bot: I'm really sorry to hear that you're stuck in such a challenging situation. Here are some steps you can take while waiting for help:

1. **Safety First**: Make sure your vehicle is not at risk of flooding further. Turn on your hazard lights and pull over if possible, away from the water.

2. **Stay Calm**: It’s important to stay calm so you don't make unnecessary mistakes or decisions that could worsen the situation.

3. **Communicate for Help**: Use your phone to call emergency services (like 911) if you're in a safe location. Alternatively, try to find someone who might be able to help from nearby—another car, a shop, or a building far enough away from the flood.

4. **Signal Your Need for Help**: I